# CoverPay - How the Machine Learning Works

CoverPay is an **advisory** fraud-intelligence layer for an Indian payment gateway.
It never blocks a payment. It scores transactions, explains the score, watches
customer behaviour and merchant-wide activity, and hands an analyst evidence.

This notebook is the guided tour of the ML. It is meant to be **read top to
bottom**; the code cells are runnable but optional, and each is guarded so a
missing artifact or dataset prints a note instead of crashing.

---

## The one idea to take away

CoverPay runs **three detectors**, and the whole design follows from the fact
that they answer three *different* questions:

| Engine | Code | Question it answers | Scope | Method |
|---|---|---|---|---|
| **Model** | `ml/`, `fraud_engine/advisory.py` | *How suspicious is this one transaction?* | per transaction | XGBoost classifier + TreeSHAP |
| **Behaviour** | `fraud_engine/behavioural.py` | *Is this unusual for **this customer**?* | per customer, over their history | robust statistics on a causal history |
| **Incident** | `fraud_engine/incidents.py` | *Is a coordinated attack hitting **this merchant**?* | per merchant, over rolling windows | rate-based change detection |

Each engine has a blind spot that another engine covers. The model reacts to the
*shape* of a transaction but knows nothing about the customer. The behaviour
engine sees a customer drift but, by construction, cannot see twenty different
customers hit one merchant at once. The incident engine sees that spike but
cannot tell an attack from a flash sale. None of them is trusted alone - they
produce **evidence**, and a separate advisory layer (and ultimately a human)
decides what to do.

## Contents

1. [Setup](#setup)
2. [The data: IEEE-CIS](#data)
3. [Phase 1 - the data audit](#audit)
4. [Phase 2 - the one feature pipeline](#features)
5. [Phase 3 - training the XGBoost model](#training)
6. [Phase 4 - inference and TreeSHAP explanations](#inference)
7. [The advisory layer (policy, not model)](#advisory)
8. [Engine 2 - the behaviour engine](#behaviour)
9. [Engine 3 - the incident / spike engine](#incidents)
10. [Putting the three engines together](#together)
11. [The AI merchant assistant (not a classifier)](#assistant)
12. [The simulator](#simulator)
13. [Honest limitations](#limits)
14. [Reproduce everything](#reproduce)

<a id="setup"></a>
## 1. Setup

The Python package lives in `backend/`. The dataset, model artifacts and docs
sit one level up (they are shared with the frontend). This cell puts `backend/`
on the import path so the real modules can be imported - nothing in this
notebook re-implements the pipeline, it calls it.

In [ ]:
import sys
from pathlib import Path

# Works whether the notebook is opened at the CoverPay root or inside backend/.
_here = Path.cwd()
BACKEND = _here / "backend" if (_here / "backend").is_dir() else _here
sys.path.insert(0, str(BACKEND))

import numpy as np
import pandas as pd

from config import ROOT, settings

print("repo root  :", ROOT)
print("backend    :", BACKEND)
print("model path :", (ROOT / settings.model_path))
print("model present    :", (ROOT / settings.model_path).exists())
print("raw IEEE-CIS dir :", settings.raw)
print("raw data present :", settings.raw.exists())

<a id="data"></a>
## 2. The data: IEEE-CIS

The model is trained on the public **IEEE-CIS Fraud Detection** dataset
(a Vesta / Kaggle release). Numbers below are from `docs/audit.md`, produced by
`python -m ml.audit`.

| Property | Value | Why it matters |
|---|---|---|
| Rows | **590,540** | one row per transaction |
| Columns | **434** | 394 transaction + 41 identity, joined on `TransactionID` |
| Fraud rate | **3.5%** (20,663 rows) | imbalance ratio **1 : 28** |
| Identity coverage | **24.4%** | most rows have no device/identity data - *missing is itself signal* |
| Time span | **182 days** | `TransactionDT` is a **seconds offset**, not a wall-clock timestamp |
| Amount | 0.25 to 31,937, median 68.77 | heavily right-skewed -> a `log_amount` feature |

### Why not "accuracy"?

At a 3.5% base rate, a model that predicts *"never fraud"* is **96.5% accurate**
and catches nothing. So CoverPay reports **PR-AUC** (area under the
precision-recall curve) as the headline number, with ROC-AUC alongside.

### The column families

The 339 `V*` columns are engineered Vesta features (counting, distance, match
flags) and are heavily redundant. `C*` are counts, `D*` are timedeltas, `M*` are
match flags, `id_*` are identity/device attributes, `card1..6` / `addr1..2` are
the instrument and address. Feature selection (next section) is entirely
data-driven from the audit - nothing is hand-picked.

<a id="audit"></a>
## 3. Phase 1 - the data audit  (`ml/audit.py`)

The audit runs **once**, reads the raw CSVs, and writes both a human report
(`docs/audit.md`) and a machine-readable `ml/feature_columns.json` that every
later phase consumes. Nothing downstream re-derives a column decision.

What it decides:

- **Drop >90%-null columns** (12 of them). XGBoost tolerates NaN, but a column
  that is *almost entirely* NaN is noise.
- **Drop near-constant columns** (30 of them: one value in >99% of rows).
- **Collapse correlated `V*` pairs.** 593 pairs correlate at >= 0.95; keeping one
  of each pair drops **127** columns. Correlation is estimated on a 100k-row
  sample because `corr()` on 339 float64 columns over 590k rows needs ~1.6 GB.
- **Choose the split points.** The data is sorted by time and cut at the 60th and
  80th percentiles of `TransactionDT` into **train / validation / test**.

### Why a *chronological* split, and why *three* sets

A **random** split lets a card's history appear on both sides of the boundary.
The model then "recognises" cards it has effectively already seen and every
reported number is inflated. A time split mimics production: train on the past,
predict the future.

Two sets is not enough either. If validation is used for **early stopping** *and*
**threshold tuning**, then anything measured on it is already optimised-against.
So CoverPay holds a **third** set - `test` - untouched until the model and the
thresholds are frozen, then measures it exactly once. The cost is real
(fewer training rows, a staler model); the benefit is a number that means
something.

> Fraud rate stays stable across the split: train 3.38%, validation 3.90%,
> test 3.44% - so the split did not accidentally concentrate fraud in one era.

In [ ]:
# The audit's machine-readable output - the single source of column decisions.
import json

cols_path = BACKEND / "ml" / "feature_columns.json"
if cols_path.exists():
    decisions = json.loads(cols_path.read_text())
    print("rows audited        :", f"{decisions['rows_audited']:,}")
    print("validation boundary :", f"{decisions['split_dt']:,.0f}  (TransactionDT)")
    print("test boundary       :", f"{decisions['test_dt']:,.0f}")
    print("columns kept        :", len(decisions["keep"]))
    print("of which categorical:", len(decisions["categorical"]),
          "->", decisions["categorical"])
    print("dropped >90% null   :", decisions["dropped_high_null"])
    print("dropped near-const  :", len(decisions["dropped_near_constant"]))
    print("dropped correlated  :", len(decisions["dropped_correlated"]))
else:
    print("feature_columns.json not found - run:  cd backend && python -m ml.audit")

<a id="features"></a>
## 4. Phase 2 - the one feature pipeline  (`ml/features.py`)

> **The rule:** training and inference call the **same** `transform()`. If the
> two ever diverge, the model looks perfect offline and scores garbage in
> production. This module is the *only* place a feature is defined.

`FeatureSpec` is the fitted state that inference must reproduce exactly - the
column order, the frequency maps, the split point. It is saved next to the model
as `models/feature_spec.json`; **a model without its spec cannot be served.**

### What `transform()` does

1. **`derive()`** - pure, no fitted state, safe on a single row:
   - `log_amount = log1p(TransactionAmt)` - tames the right skew.
   - `hour_of_day`, `day_of_week` from the `TransactionDT` offset. The raw offset
     is **excluded** as a feature - it would let the model memorise the
     train/test boundary - but time-of-day within it is legitimate signal.
   - email provider = first token of the domain (`gmail.com` -> `gmail`),
     collapsing the long tail without a hand-maintained lookup.

2. **Frequency encoding** for high-cardinality identifiers (`card1`, `addr1`,
   `P_emaildomain`, ...). The integer value of `card1` is an arbitrary label, but
   *how often it appears* is strong signal: a card seen once in 590k rows behaves
   very differently from one seen 5,000 times. Each such column becomes a
   `<col>_freq` count. An unseen value maps to **0** - itself informative, and it
   must never raise at inference time.

3. **Missing stays missing.** Only 24% of rows carry identity data. Rather than
   impute, the NaN is passed straight to XGBoost, which learns a default
   direction for it. Missingness is treated as signal.

4. The frame is assembled as **whole blocks and concatenated once** - assigning
   ~270 columns one at a time fragments the DataFrame, and this runs on every
   scoring request.

### The leakage guard

`fit()` is called on **training rows only**. Fitting the frequency maps on the
full dataset would leak *future* card frequencies backwards into the training
features. There is also a loud check: if a non-numeric column slips into the
numeric list it would silently become an all-NaN column (a real signal lost), so
`fit()` raises instead.

In [ ]:
# derive() is pure - you can watch it work on a single hand-built transaction.
from ml.features import derive

one = pd.DataFrame([{
    "TransactionID": 3545324,
    "TransactionDT": 12_500_000,     # a seconds offset ~ day 144
    "TransactionAmt": 4999.0,
    "ProductCD": "C",
    "card1": 13926, "card4": "visa", "card6": "credit",
    "P_emaildomain": "gmail.com",
    "C1": 1.0, "C13": 0.0,
}])

d = derive(one)
print("log_amount  :", round(float(d["log_amount"][0]), 4), " (= log1p(4999))")
print("hour_of_day :", int(d["hour_of_day"][0]), " (= (12_500_000 // 3600) % 24)")
print("day_of_week :", int(d["day_of_week"][0]))
print("email token :", d["P_emaildomain_provider"][0])

In [ ]:
# The full pipeline: raw row -> model matrix. Same code path as training.
from ml.features import transform
from ml.inference import get_model, ModelUnavailable

try:
    model, spec = get_model()          # loads booster + FeatureSpec, cached
    X = transform(one, spec)
    print("model matrix shape :", X.shape, " (1 row x", len(spec.columns), "features)")
    print("dtype              :", X.dtypes.unique().tolist(), " <- XGBoost needs numbers")
    print("first few columns  :", spec.columns[:6])
    print("a frequency column :",
          [c for c in spec.columns if c.endswith('_freq')][:5])
except ModelUnavailable as exc:
    print("Model artifacts missing:", exc)

<a id="training"></a>
## 5. Phase 3 - training the model  (`ml/train.py`)

### Why XGBoost (gradient-boosted decision trees)

| Requirement | Why a GBDT fits |
|---|---|
| Wide, messy **tabular** data (270 mixed-scale columns) | trees are scale-free; no normalisation, no encoding pipeline to drift |
| **NaN everywhere** (76% of `id_*` missing) | XGBoost learns a default split direction for missing values natively |
| Severe **class imbalance** (1:28) | `scale_pos_weight` re-weights the gradient toward the rare class |
| Needs **explanations** per prediction | exact **TreeSHAP** comes free from the booster (`pred_contribs=True`) - no extra library, no surrogate model |
| Trains on a laptop in **minutes** | `tree_method="hist"` bins features once |

A linear model would need heavy preprocessing and still miss the interactions.
A neural net is overkill for 270 tabular columns and gives no cheap exact
attributions. Deep learning is not the right tool for this data shape.

### Hyperparameters (`PARAMS` in `ml/train.py`)

```python
n_estimators        = 2000     # upper bound; early stopping picks the real count
learning_rate       = 0.05
max_depth           = 8
subsample           = 0.8      # row sampling per tree - regularisation
colsample_bytree    = 0.8      # column sampling per tree
min_child_weight    = 4        # don't split on a handful of rows
tree_method         = "hist"
eval_metric         = "aucpr"  # PR-AUC, not accuracy - the classes are 1:28
early_stopping_rounds = 100    # stop when validation PR-AUC stalls
scale_pos_weight    = (#neg / #pos) ~ 28
```

### The flow inside `main()`

```text
load_raw()  ->  split(train | rest)  ->  split(rest -> val | test)
                     |
       fit(FeatureSpec) on TRAIN ROWS ONLY
                     |
   transform() train / val / test with that one spec
                     |
   XGBClassifier.fit(train, eval_set=[val], early_stopping)
                     |
   sweep 19x19 (review, block) threshold pairs  ON VALIDATION  -> cost-optimal pair
                     |
   score TEST once  ->  docs/metrics.md  +  models/fraud_xgb.json  +  feature_spec.json
```

The raw frame is ~1.8 GB and is `del`-eted and garbage-collected right after
`transform()`, before training, so a 16 GB machine does not swap.

### 5.1 What the trained model actually scores  (`docs/metrics.md`)

All numbers below are measured **once, on the held-out test set** (118,108 rows,
3.44% fraud).

| Metric | Value | Reading |
|---|---|---|
| **PR-AUC** | **0.4746** | vs a random baseline of 0.0344 -> **13.8x lift** |
| ROC-AUC | 0.8681 | ranking quality overall |
| Best iteration | 490 of 2000 | early stopping stopped here |
| Features | 273 | after audit pruning + derived columns |

**Top features by gain:** `V258`, `V201`, `V69`, `V90`, `V283`, ... then `C8`,
`C14`, `C11`, and `card6_freq` / `ProductCD_freq` (the frequency-encoded
columns earning their place).

### The cost model - precision/recall in rupees

A threshold is a *business* decision, and F1 is the wrong unit for arguing about
one. `ml/train.py` converts the confusion matrix into money using three
**assumptions** that live in `.env` (not measurements):

| Input | Default | Meaning |
|---|---|---|
| `CHARGEBACK_FEE` | 750 | admin cost of a chargeback, on top of the lost amount |
| `REVIEW_COST` | 40 | analyst cost to clear one flagged transaction |
| `BLOCK_FRICTION_RATE` | 0.25 | share of a wrongly-BLOCKed legit amount that walks away |

> Cost of a policy = missed-fraud (amount + fee) + analyst time on everything
> flagged + friction on legit transactions that reached BLOCK.

| Policy | Review / Block | Net cost | Saved vs doing nothing |
|---|---|---|---|
| Doing nothing | - | 3,657,934 | - |
| Configured (`.env`) | 0.30 / 0.70 | **1,862,064** | 1,795,870  (49.1%) |
| Cost-optimal on validation | 0.25 / 0.95 | **1,799,925** | 1,858,010  (50.8%) |

Thresholds are policy: changing them in `.env` needs **no retraining**.

<a id="inference"></a>
## 6. Phase 4 - inference and explanations  (`ml/inference.py`)

`get_model()` loads the booster and its `FeatureSpec` **once per process**
(`lru_cache`) - reloading per request would dominate latency. If either artifact
is missing it raises `ModelUnavailable`, which the API turns into a `503` rather
than scoring with a silently-untrained model.

### Every score arrives with its reasons

The score is never returned alone. CoverPay asks the booster for **exact
TreeSHAP** contributions via `booster.predict(dmatrix, pred_contribs=True)`.
This returns, for each row, one contribution per feature **plus a bias term**
(the last element - the model's average output, constant across transactions, so
*not* evidence).

`reasons_for_row()` is the **single definition** of "which features are the
reasons": it drops the bias, takes the top-5 by absolute contribution, and
records the feature, its value, and the signed contribution. `score()`,
`score_batch()` and the simulator feed all call it - so a transaction's **stored**
evidence can never be selected differently from its **live** evidence.

```python
per_feature = contribs[:-1]                       # drop the bias term
top = np.argsort(-np.abs(per_feature))[:5]        # 5 biggest movers, either direction
```

Positive contribution -> "increases risk"; negative -> "decreases risk". Values
are formatted for a human reading a queue (a six-digit card number prints in
full, not as `9.06e+05`).

### `score()` vs `score_batch()`

- **`score(one)`** - a single transaction. One `predict_proba`, one TreeSHAP row.
- **`score_batch(df)`** - a CSV upload. The probability pass is vectorised and
  **every row is scored** (the caller gets counts for the whole file, never a
  silent prefix). A full contributions matrix is O(rows x trees x features) and
  does not scale, so exact TreeSHAP is computed only for the **`shap_top_n`
  riskiest** rows; the rest carry a one-line summary reason. `max_rows` is a
  memory rail - above it the batch is **rejected**, not quietly truncated to a
  length the totals would then misreport.

In [ ]:
# Score one transaction end to end, with its evidence.
from ml.inference import score, ModelUnavailable

sample = {
    "TransactionID": 9999001,
    "TransactionDT": 12_500_000,
    "TransactionAmt": 4999.00,
    "ProductCD": "C",              # C has the highest fraud rate in the audit
    "card1": 15066, "card4": "visa", "card6": "credit",
    "P_emaildomain": "protonmail.com",   # an unseen provider -> _freq maps to 0
    "C1": 3.0, "C13": 1.0, "C14": 1.0,
}

try:
    r = score(sample)
    print(f"transaction    {r.transaction_id}")
    print(f"risk_score     {r.risk_score:.4f}")
    print(f"risk_level     {r.risk_level}")
    print(f"recommendation {r.recommendation}")
    print(f"amount         {r.amount:,.2f}")
    print("reasons (top 5 TreeSHAP contributors):")
    for reason in r.reasons:
        print(f"  - {reason.text}   ({reason.contribution:+.3f})")
except ModelUnavailable as exc:
    print("Model artifacts missing:", exc, "\nRun:  cd backend && python -m ml.train")

In [ ]:
# Look directly at the raw contributions vector, including the bias term.
import xgboost as xgb
from ml.inference import get_model, ModelUnavailable
from ml.features import transform

try:
    model, spec = get_model()
    X = transform(pd.DataFrame([sample]), spec)
    contribs = model.get_booster().predict(
        xgb.DMatrix(X, feature_names=spec.columns), pred_contribs=True
    )[0]
    print("contribs length :", len(contribs), "=", len(spec.columns), "features + 1 bias")
    print("bias term       :", round(float(contribs[-1]), 4), " <- constant, not evidence")
    per_feature = contribs[:-1]
    order = np.argsort(-np.abs(per_feature))[:8]
    print("\n  feature            value        contribution")
    for i in order:
        print(f"  {spec.columns[i]:<18} {float(X.iloc[0, i]):>10.3f}   {per_feature[i]:+.4f}")
    print("\nsum(all contribs) ~ model's raw (logit) output for this row:",
          round(float(contribs.sum()), 4))
except ModelUnavailable as exc:
    print("Model artifacts missing:", exc)

<a id="advisory"></a>
## 7. The advisory layer  (`fraud_engine/advisory.py`)

Deliberately a separate 40-line module. The model answers *"how suspicious?"*;
policy answers *"what should the merchant do?"*.

```python
score >= RISK_BLOCK_THRESHOLD  (0.70)  ->  HIGH,   BLOCK
score >= RISK_REVIEW_THRESHOLD (0.30)  ->  MEDIUM, REVIEW
otherwise                              ->  LOW,    ALLOW
```

- `BLOCK` is **advisory only** - CoverPay never stops a real payment. It is a
  strong recommendation to a merchant/analyst.
- The thresholds are `.env` config, tunable per merchant risk appetite, and
  changing them retrains nothing.
- `classify()` validates that the score is a probability in `[0, 1]` and raises
  otherwise - a bad score should fail loudly, not silently land in a band.

<a id="behaviour"></a>
## 8. Engine 2 - the behaviour engine  (`fraud_engine/behavioural.py`)

> The model asks *"does this look like fraud?"*. This asks what the model
> cannot: *"is this unusual **for this customer**?"* A 4,000-rupee payment is
> unremarkable in general and alarming from someone who has never spent above 300.

### Strictly causal

Each transaction is judged against **only the transactions that came before
it**; state is updated *afterwards*. Peeking at a customer's full history
(including their future) is the behavioural equivalent of a random train/test
split - it would make every number here a lie. A customer needs `MIN_HISTORY = 3`
prior payments before "unusual for them" means anything.

### The three signals

**1. Amount anomaly** - a **robust** z-score.

```python
median = median(history.amounts)
mad    = median(|a - median| for a in history.amounts)      # median absolute deviation
z      = |amount - median| / (mad * 1.4826)                  # 1.4826 makes MAD ~ sigma
flag if z >= 3.5
```

Median/MAD, **not** mean/standard-deviation, because a single huge fraud would
inflate the standard deviation and *hide itself*. Severity scales with `z`,
capped at 1.0.

**2. Velocity** - `VELOCITY_THRESHOLD = 5` prior transactions from one customer
inside a 300-second window. Bunched payments look automated. The window is pruned
as time advances (a `deque` of recent `TransactionDT` values).

**3. New device** - a device never seen for a customer who already has an
established pattern (`>= 3` prior payments). Fixed severity 0.6.

### Combining signals - noisy-OR, not a sum

```python
remaining = 1.0
for s in signals:
    remaining *= 1.0 - WEIGHT[s.name] * s.severity     # amount .45, velocity .40, new_device .35
score = 1.0 - remaining
```

Two independent weak signals should raise suspicion **without** three of them
mechanically pinning the score at exactly 1.0.

### It is entity-scoped *on purpose*

Signals concern one customer or one device. Merchant-wide, cross-entity patterns
are explicitly *not* this engine's job - that is Engine 3.

In [ ]:
# A hand-built stream so you can see exactly what fires.
# One customer (CUST_A) with a calm history, then a card-testing burst.
from fraud_engine.behavioural import analyse_stream

rows = []
t = 1_000_000
# CUST_A: 6 ordinary payments around 200, one per ~hour
for i in range(6):
    rows.append({"customer_id": "CUST_A", "device_id": "DEV_A", "merchant_id": "M1",
                 "TransactionDT": t, "TransactionAmt": 200 + i * 7})
    t += 3600
# CUST_A: sudden burst of 8 tiny payments in ~90s  -> velocity should fire
for _ in range(8):
    t += 12
    rows.append({"customer_id": "CUST_A", "device_id": "DEV_A", "merchant_id": "M1",
                 "TransactionDT": t, "TransactionAmt": 3.50})
# CUST_A: one big-ticket payment from a brand-new device -> amount + new_device
rows.append({"customer_id": "CUST_A", "device_id": "DEV_STOLEN", "merchant_id": "M9",
             "TransactionDT": t + 400, "TransactionAmt": 9000.0})

stream = pd.DataFrame(rows)
analysed = analyse_stream(stream)
out = stream.join(analysed)[["TransactionDT", "TransactionAmt", "device_id",
                             "behaviour_score", "behaviour_signals", "behaviour_detail"]]
pd.set_option("display.max_colwidth", 60)
print(out.to_string(index=False))

### 8.1 How the behaviour engine actually performs  (`docs/behaviour.md`)

Measured on a 4,000-transaction seeded synthetic stream (threshold 0.3). **The ML
model is not involved** - these are behavioural signals only.

| Metric | Value |
|---|---|
| Precision | **0.890** |
| Recall | 0.333 |
| False positives | 83 of 1,985 legitimate |

| Scenario | Flag rate | Comment |
|---|---|---|
| `card_testing` | **0.537** | caught by **velocity** (fires 866x on fraud, **0x** on normal) |
| `account_takeover` | **0.336** | caught by **new_device** + **amount_anomaly** |
| `coordinated_spike` | **0.082** | *expected to fail here* - see below |
| `normal` | 0.042 | the false-positive rate |

The low **recall** is honest and expected: many injected "fraud" rows are the
quiet middle of a burst, and the engine only needs to flag the burst, not every
row of it. `coordinated_spike` scores ~8% because each of its transactions is
**ordinary for the individual customer** - the pattern only exists *across*
customers at one merchant. That is precisely the gap Engine 3 fills.

<a id="incidents"></a>
## 9. Engine 3 - the incident / spike engine  (`fraud_engine/incidents.py`)

> Twenty customers each making one ordinary-looking payment at the same merchant
> in five minutes is unremarkable per customer and obvious in aggregate. This
> engine watches **merchants** over rolling windows.

### Rate-based, not suspicion-based

A spike is a spike whether or not the individual transactions look fraudulent -
which is exactly why it catches what the other two miss. It is also why the
output is **advisory evidence for an analyst**: a rate detector cannot tell an
attack from a genuine surge (a flash sale looks identical).

### Two rolling windows per merchant

```python
WINDOW_SECONDS   = 300     # "what is happening now"
BASELINE_SECONDS = 3600    # "what this merchant normally does"
```

Both are `deque`s, pruned as time advances. Detection fires when the short window
has at least `MIN_WINDOW_COUNT = 8` transactions **and** either:

- **rate ratio** `>= 4.0` - `(window_count / 300) / (baseline_count / 3600)`
  (the baseline includes the window, which makes the ratio conservative, not
  flattering; a finite `RATE_RATIO_CAP` keeps it JSON-safe), **or**
- **customer fanout** `>= 6` - distinct customers in the window (the
  coordinated-attack fingerprint).

### Incident lifecycle

```text
NORMAL -> SUSPICIOUS -> ACTIVE -> RESOLVED
```

A new spike opens a `SUSPICIOUS` incident. It becomes `ACTIVE` at
`ACTIVE_COUNT = 20` transactions. After `COOLDOWN_SECONDS = 900` of quiet it is
`RESOLVED`. While open it accumulates: transactions, distinct customers/devices,
total and "risky" value (risky = the behaviour engine's score cleared a
threshold), peak rate ratio, peak fanout.

### Severity - blends *how abnormal*, *how broad*, *how much money*

```python
score  = (peak_rate_ratio >= 8) + (peak_fanout >= 12)
       + (transactions >= 20)   + (risky_value > 10_000)
severity = [LOW, MEDIUM, HIGH, CRITICAL][min(score, 3)]
```

In [ ]:
# Feed a coordinated spike through the incident engine.
from fraud_engine.incidents import detect_incidents

rows = []
t = 2_000_000
# MERCH_X ticks along quietly: ~1 txn / 5 min for an hour (builds a baseline)
for i in range(12):
    rows.append({"merchant_id": "MERCH_X", "customer_id": f"reg_{i}", "device_id": f"d{i}",
                 "TransactionDT": t, "TransactionAmt": 500.0})
    t += 300
# then 25 different customers hit MERCH_X inside ~4 minutes
for i in range(25):
    t += 10
    rows.append({"merchant_id": "MERCH_X", "customer_id": f"victim_{i}", "device_id": f"v{i}",
                 "TransactionDT": t, "TransactionAmt": 1800.0})

stream = pd.DataFrame(rows)
per_row, incidents = detect_incidents(stream)

print("rows now inside a detected incident:",
      int((per_row["incident_id"] != "").sum()), "of", len(stream))
print("peak rate_ratio seen   :", round(per_row["rate_ratio"].max(), 1))
print("peak customer_fanout   :", int(per_row["customer_fanout"].max()))
print()
for inc in incidents:
    s = inc.summary()
    print(f"{s['incident_id']}  merchant={s['merchant_id']}  status={s['status']}  "
          f"severity={s['severity']}")
    print(f"   {s['transactions']} txns, {s['customers']} customers, "
          f"value {s['total_value']:,.0f}, peak_rate_ratio {s['peak_rate_ratio']}, "
          f"peak_fanout {s['peak_fanout']}")

### 9.1 How the incident engine performs  (`docs/incidents.md`)

Same 4,000-transaction seeded stream.

| Metric | Value |
|---|---|
| Incident recall | **0.629** (73 of 116 injected) |
| Transaction-level precision | **0.973** |
| Incidents raised | 71 |

| Scenario | Incident recall | Comment |
|---|---|---|
| `coordinated_spike` | **1.000** | this is exactly what it is for |
| `card_testing` | **0.969** | a burst from one card is also a merchant-level rate spike |
| `account_takeover` | **0.000** | *expected* - 2-6 payments from one customer is not a merchant spike; that is Engine 2's catch |

Severity mix: 8 CRITICAL, 45 HIGH, 14 MEDIUM, 4 LOW.

### The symmetry - each engine covers another's blind spot

| Scenario | Model | Behaviour | Incident |
|---|---|---|---|
| `card_testing` (automated burst) | weak (reacts to template) | **0.54** (velocity) | **0.97** (rate spike) |
| `account_takeover` (new device, big spend) | weak | **0.34** (new_device + amount) | 0.00 |
| `coordinated_spike` (many customers, one merchant) | weak | 0.08 | **1.00** |

No single row of that table is good enough. Together they cover the space.

<a id="together"></a>
## 10. Putting the three engines together  (`simulator/feed.py`)

`python -m simulator.feed --rows 4000 --reset` runs the exact pipeline the design
doc draws end to end:

```text
simulator ->  model (score + TreeSHAP)
          ->  behaviour engine (per-customer signals)
          ->  incident engine (per-merchant spikes, using behaviour_score as "risky")
          ->  SQLite  (Transaction, Prediction, IncidentRecord)
```

Then the FastAPI service serves it and the React dashboard reads it.

### Two deliberate boundaries

1. **The dashboard reads only the database.** It never imports a model or a
   simulator. The read path stays fast, and the dashboard would work identically
   against real scored traffic.

2. **Ground truth never enters the database.** `synthetic_is_fraud` and the
   `synthetic_scenario_*` columns are *evaluation metadata*. Writing them would
   let a dashboard quietly display answers no real deployment could know.

### Evidence aggregation

Each `Prediction` row stores: `risk_score` / `risk_level` / `recommendation`
(model), `reasons` (the top-5 TreeSHAP JSON), `behaviour_score` +
`behaviour_detail`, and `incident_id` (link to any incident the transaction
belongs to). The advisory recommendation on the record is the model's; the
behaviour and incident columns sit beside it so an analyst - or the assistant -
sees all three engines at once.

<a id="assistant"></a>
## 11. The AI merchant assistant  (`api/assistant.py`)

**The LLM is not the fraud classifier and has no authority over any fraud
decision.** Its only job is to explain, in plain language, evidence the three
engines already produced.

### How it stays grounded

1. A merchant question (e.g. *"why did my fraud risk go up today?"*) triggers
   **SQL queries** against the stored transactions, predictions, behaviour
   signals and incidents for that merchant and time window.
2. Those query results - and *only* those - are serialised into a `context`
   block and sent to Gemini with a system prompt that says: use these facts,
   nothing else.
3. The same `context` is returned in the API response, so the answer can be
   **checked against its evidence** rather than trusted.

### Prompt-injection handling

The merchant's text is wrapped in delimiters and the system prompt states that
everything between them is a *question to answer, never an instruction*. The
literal closing delimiter is stripped from the user's text so it cannot break
out of the wrapper.

### Key not in code

The Gemini API key is read from `.env` (gitignored) via `pydantic-settings`,
never hard-coded and never sent to the frontend.

<a id="simulator"></a>
## 12. The simulator  (`simulator/world.py`, `simulator/generate.py`)

The IEEE-CIS data has no entity identifiers - every card appears once, so there
is no *history*. The behaviour and incident engines need history. The simulator
builds a synthetic **stream**: a fixed population (~400 customers, 12 merchants,
devices) whose transactions recur over time, so entities accumulate a past.

### Two modes

- **`evaluation`** - seeded from `settings.random_seed`, reproducible, known
  ground truth. This is what every `docs/*.md` evaluation uses.
- **`live`** - a fresh seed each run, randomly timed scenarios, for demos.

### Three injected scenario families (matching the PRD's fraud classes)

| Scenario | Shape |
|---|---|
| `card_testing` | one card, 15-45 tiny payments in minutes (automated probe) |
| `account_takeover` | an established customer, sudden new device, 6-15x their usual spend |
| `coordinated_spike` | 10-30 unrelated customers hitting one merchant in a short window |

### The honest limit

The ~380 wide `V/C/D/M/id_` columns are **borrowed from real IEEE-CIS template
rows** - no generator can invent their joint structure. So the **ML model reacts
mostly to the borrowed template, not to the scenario shape**. The synthetic
scenarios are built for the behaviour and incident engines, which read entity
history rather than `V*` columns. A synthetic stream is *not* evidence that the
ML model detects these attacks - the model's evidence is `docs/metrics.md`, on
real held-out IEEE-CIS data.

Every `synthetic_*` column is evaluation metadata and is **never a model input**.

<a id="limits"></a>
## 13. Honest limitations

| Area | Limitation |
|---|---|
| **Model ceiling** | Test PR-AUC 0.47. Good lift (13.8x) but far from solved - IEEE-CIS is hard, and the chronological three-way split deliberately reports a *lower* number than the usual two-way split would. |
| **Simulator** | Wide feature columns are borrowed from real templates, so the ML model does not meaningfully "see" the synthetic scenarios (section 12). |
| **Behaviour engine** | Amount stats are recomputed per transaction (fine at tens of payments per customer; a streaming quantile would be needed for long histories). Thresholds (`Z=3.5`, velocity 5/300s) are hand-set policy, not learned. |
| **Incident engine** | Rate-based -> cannot distinguish an attack from a flash sale. Advisory only, by design. |
| **Identity** | A valid API key does **not** identify a merchant. There is no principal->merchant model, so `merchant_id` filtering is not yet trustworthy for multi-tenant use. |
| **Infra** | In-process fixed-window rate limiter (resets on restart, not shared across workers); SQLite with WAL (Postgres migration documented, not done). |
| **Assistant** | Depends on an external API (Gemini); grounded but still a generative model - the returned `context` is the check. |

None of these are hidden - each is a `ponytail:` comment or a paragraph in the
relevant `docs/*.md`.

<a id="reproduce"></a>
## 14. Reproduce everything

From `backend/`, with the IEEE-CIS CSVs in the folder named by `DATA_DIR` in
`.env`:

```bash
cd backend
python -m venv .venv && .venv\Scripts\activate      # Windows
pip install -e ".[dev]"

# --- the model ---
python -m ml.audit          # -> docs/audit.md, ml/feature_columns.json
python -m ml.train          # -> models/fraud_xgb.json, models/feature_spec.json, docs/metrics.md
python -m ml.inference      # smoke test: scores the worst fraud in validation, checks batch==single

# --- the other two engines (synthetic stream, seeded) ---
python -m fraud_engine.behavioural --rows 4000     # -> docs/behaviour.md
python -m fraud_engine.incidents   --rows 4000     # -> docs/incidents.md

# --- everything into the database, then serve ---
python -m simulator.feed --rows 4000 --reset
uvicorn api.main:app --reload                       # http://127.0.0.1:8000/docs
```

### Where each piece lives

```text
backend/
  ml/audit.py           Phase 1  - data audit, column decisions
  ml/features.py        Phase 2  - the one feature pipeline (fit / transform / split)
  ml/train.py           Phase 3  - training + cost model + docs/metrics.md
  ml/inference.py       Phase 4  - get_model, score, score_batch, reasons_for_row (TreeSHAP)
  fraud_engine/advisory.py       - score -> LOW/MEDIUM/HIGH + ALLOW/REVIEW/BLOCK
  fraud_engine/behavioural.py    Engine 2 - per-customer anomaly (causal)
  fraud_engine/incidents.py      Engine 3 - per-merchant spike + incident lifecycle
  simulator/world.py             synthetic entity-history stream + 3 scenarios
  simulator/generate.py          independent-row generator + template pools
  simulator/feed.py              Phase 13 - stream -> all engines -> SQLite
  api/                           FastAPI: /transactions/score, /assistant/ask, dashboard
models/  fraud_xgb.json, feature_spec.json     the served artifacts (spec is mandatory)
docs/    audit.md, metrics.md, behaviour.md, incidents.md, fraud_prd.md, ...engineering_design.md
```